# Notebook de Inspección de Datos – Contrataciones Públicas Perú
## Mes de muestra: diciembre 2025 (12-2025)

**Contexto:**  
Disponemos de 6 archivos CSV por mes, siguiendo el estándar OCDS. En este notebook cargaremos un mes de muestra (carpeta `usar/12-2025/`) y realizaremos un análisis exploratorio profundo para:
- Conocer la estructura, tipos y calidad de los datos.
- Detectar columnas con muchos nulos, duplicados, posibles IDs, valores atípicos.
- Tomar decisiones sobre limpieza y normalización (RUC, fechas, texto).

In [1]:
# Imports y configuración global
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Configuración de rutas – Usaremos la carpeta `usar/12-2025/` como muestra

In [3]:
# Definir rutas
# Carpeta base donde están los datos (relativa al notebook)
ruta_base = Path("./usar")

# Mes de muestra (diciembre 2025)
mes_muestra = "12-2025"
ruta_muestra = ruta_base / mes_muestra

# Verificar que la carpeta existe
if ruta_muestra.exists():
    print(f"Carpeta de muestra encontrada: {ruta_muestra.resolve()}")
else:
    raise FileNotFoundError(f"No se encontró la carpeta {ruta_muestra}. Verifica la estructura.")

# Listar todos los meses disponibles (por si queremos después)
meses_disponibles = sorted([d.name for d in ruta_base.iterdir() if d.is_dir()])
print(f"Meses disponibles (total: {len(meses_disponibles)}): {meses_disponibles[:5]} ...")

Carpeta de muestra encontrada: C:\Users\nicol\OneDrive\Desktop\DESARROLLO\datos_seace\datos\usar\12-2025
Meses disponibles (total: 1): ['12-2025'] ...


### 2. Funciones auxiliares
- `leer_csv_robusto()`: prueba varios encodings para leer cada archivo.
- `inspeccion_profunda()`: genera reporte completo de calidad.

In [5]:
# Función de lectura robusta
def leer_csv_robusto(ruta, dtype_cols=None):
    """Intenta leer un CSV con varios encodings (utf-8, latin1, cp1252)."""
    encodings = ['utf-8', 'latin1', 'cp1252']
    for enc in encodings:
        try:
            df = pd.read_csv(ruta, encoding=enc, dtype=dtype_cols, low_memory=False)
            print(f"✓ Leído {ruta.name} con encoding {enc} – {df.shape[0]:,} filas, {df.shape[1]} columnas")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"✗ Error con {enc} en {ruta.name}: {e}")
    print(f"✗ No se pudo leer {ruta.name} con ningún encoding.")
    return None

In [9]:
def inspeccion_profunda(df, nombre_archivo):
    if df is None:
        print(f"No se puede inspeccionar '{nombre_archivo}' (DataFrame vacío).")
        return

    # ============================================================
    # CABECERA
    # ============================================================
    print("\n" + "="*80)
    print(f"INSPECCIÓN: {nombre_archivo}")
    print("="*80)

    # 1. Dimensiones
    print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")

    # 2. Lista de columnas y sus tipos de datos (más legible)
    print("\n COLUMNAS Y TIPOS:")
    for col in df.columns:
        print(f"   • {col} → {df[col].dtype}")

    # 3. Vista previa (primeras 3 filas)
    print("\n PRIMERAS 3 FILAS:")
    print(df.head(3).to_string(max_cols=8))  # limitamos a 8 columnas para no saturar

    # 4. Valores faltantes (solo si hay)
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df)) * 100
    nulos_df = nulos[nulos > 0].sort_values(ascending=False)
    if not nulos_df.empty:
        print("\n VALORES FALTANTES (5 primeros):")
        for col in nulos_df.head(5).index:
            print(f"   {col}: {nulos[col]} nulos ({nulos_pct[col]:.1f}%)")
    else:
        print("\n Sin valores faltantes.")

    # 5. Duplicados exactos (solo si hay)
    dup = df.duplicated().sum()
    if dup > 0:
        print(f"\n🔄 Filas duplicadas exactas: {dup} ({100*dup/len(df):.1f}%)")
    else:
        print("\n Sin filas duplicadas exactas.")

    # 6. Columnas con un solo valor (útiles para descartar)
    constantes = [col for col in df.columns if df[col].nunique() == 1]
    if constantes:
        print(f"\Columnas constantes (un solo valor): {constantes}")

    # 7. Estadística rápida de columnas numéricas (solo las más relevantes)
    num_cols = df.select_dtypes(include='number').columns
    if len(num_cols) > 0:
        print("\n RESUMEN NUMÉRICO (media, mediana, rango):")
        for col in num_cols[:5]:  # limitamos a 5 para no abrumar
            print(f"   {col}: media = {df[col].mean():.2f}, mediana = {df[col].median():.2f}, "
                  f"min = {df[col].min():.2f}, max = {df[col].max():.2f}")
    else:
        print("\n No hay columnas numéricas.")

    # 8. Ejemplo de valores únicos en columnas categóricas (primeras 5)
    cat_cols = df.select_dtypes(include='object').columns
    if len(cat_cols) > 0:
        print("\n EJEMPLOS DE VALORES EN COLUMNAS CATEGÓRICAS (primeras 5):")
        for col in cat_cols[:5]:
            unicos = df[col].dropna().unique()
            ejemplo = unicos[:3] if len(unicos) > 3 else unicos
            print(f"   {col}: {len(unicos)} únicos → ejemplos: {ejemplo}")
    else:
        print("\n No hay columnas categóricas (object).")

    # 9. Rango de fechas si hay columnas datetime
    date_cols = df.select_dtypes(include='datetime64').columns
    if len(date_cols) > 0:
        print("\n RANGO DE FECHAS:")
        for col in date_cols:
            print(f"   {col}: desde {df[col].min()} hasta {df[col].max()}")

    print("\n" + "-"*80)
    print(" Inspección finalizada.")
    print("-"*80 + "\n")

### 3. Cargar los 6 archivos del mes de muestra
Lista de archivos esperados:
- `Ent_Adj_Proveedores.csv` (proveedores ganadores)
- `Ent_Adjudicaciones.csv` (adjudicaciones)
- `Ent_Contratos.csv` (contratos)
- `Ent_Lic_Licitantes.csv` (postores)
- `Ent_PartesInvolucradas.csv` (partes involucradas)
- `Registros.csv` (licitaciones)

In [7]:
# Cargar todos los archivos de la muestra
archivos_esperados = [
    "Ent_Adj_Proveedores.csv",
    "Ent_Adjudicaciones.csv",
    "Ent_Contratos.csv",
    "Ent_Lic_Licitantes.csv",
    "Ent_PartesInvolucradas.csv",
    "Registros.csv"
]

dataframes_muestra = {}
for archivo in archivos_esperados:
    ruta_completa = ruta_muestra / archivo
    if ruta_completa.exists():
        df = leer_csv_robusto(ruta_completa)
        dataframes_muestra[archivo] = df
    else:
        print(f" No se encontró {archivo} en {ruta_muestra}")
        dataframes_muestra[archivo] = None

✓ Leído Ent_Adj_Proveedores.csv con encoding utf-8 – 6,254 filas, 5 columnas
✓ Leído Ent_Adjudicaciones.csv con encoding utf-8 – 6,264 filas, 7 columnas
✓ Leído Ent_Contratos.csv con encoding utf-8 – 5,179 filas, 17 columnas
✓ Leído Ent_Lic_Licitantes.csv con encoding utf-8 – 84,597 filas, 5 columnas
✓ Leído Ent_PartesInvolucradas.csv con encoding utf-8 – 92,905 filas, 16 columnas
✓ Leído Registros.csv con encoding utf-8 – 8,306 filas, 39 columnas


### 4. Ejecutar inspección profunda para cada tabla
Esto generará reportes detallados en la salida. Por ejemplo:
- Columnas con altos nulos (candidatas a descarte).
- Columnas con cardinalidad casi total → posibles IDs (las usaremos para joins pero no como features).
- Presencia de duplicados exactos.
- Distribución de montos y fechas.
- Valores atípicos.

In [10]:
# Inspeccionar cada DataFrame cargado
for nombre, df in dataframes_muestra.items():
    if df is not None:
        inspeccion_profunda(df, nombre)
    else:
        print(f"\n Saltando inspección de {nombre} (archivo no cargado).")


INSPECCIÓN: Ent_Adj_Proveedores.csv
Filas: 6,254  |  Columnas: 5

 COLUMNAS Y TIPOS:
   • Open Contracting ID → object
   • Entrega compilada:ID de Entrega → object
   • Entrega compilada:Adjudicaciones:ID de Adjudicación → object
   • Entrega compilada:Adjudicaciones:Proveedores:ID de Organización → object
   • Entrega compilada:Adjudicaciones:Proveedores:Nombre de la Organización → object

 PRIMERAS 3 FILAS:
           Open Contracting ID                               Entrega compilada:ID de Entrega Entrega compilada:Adjudicaciones:ID de Adjudicación Entrega compilada:Adjudicaciones:Proveedores:ID de Organización Entrega compilada:Adjudicaciones:Proveedores:Nombre de la Organización
0  ocds-dgv273-seacev3-1218759  ocds-dgv273-seacev3-1218759-2026-05-31T10:15:02.851291-05:00                                 1218759-20600588479                                              PE-RUC-20600588479                                                   GRUPO AMERINKA S.R.L
1  ocds-dgv273-seacev3-12

In [11]:
RUTA_BASE = Path("./usar")

carpetas_meses = sorted(
    [d.name for d in RUTA_BASE.iterdir() if d.is_dir()]
)
print(f"Meses encontrados ({len(carpetas_meses)}): {carpetas_meses}")

Meses encontrados (12): ['01-2025', '02-2025', '03-2025', '04-2025', '05-2025', '06-2025', '07-2025', '08-2025', '09-2025', '10-2025', '11-2025', '12-2025']


In [12]:
# Funciones Auxiliares
def extraer_ruc(org_id) -> str:
    if pd.isna(org_id):
        return np.nan
    candidato = str(org_id).split("-")[-1]
    if candidato.isdigit():
        return candidato if len(candidato) == 11 else "CONSORCIO_SIN_RUC"
    return np.nan


def limpiar_fechas(df: pd.DataFrame, columnas: list) -> pd.DataFrame:
    for col in columnas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            if pd.api.types.is_datetime64tz_dtype(df[col]):
                df[col] = df[col].dt.tz_localize(None)
    return df


def inferir_metodo(row: pd.Series) -> str:
    if pd.notna(row.get("metodo_contratacion")):
        return str(row["metodo_contratacion"]).strip()
    detalle = str(row.get("detalle_metodo", "")).upper()
    if any(k in detalle for k in ("LICITACIÓN PÚBLICA", "CONCURSO PÚBLICO")):
        return "open"
    if any(k in detalle for k in ("ADJUDICACIÓN SIMPLIFICADA", "SUBASTA")):
        return "selective"
    if any(k in detalle for k in ("CONTRATACIÓN DIRECTA", "EXONERACIÓN")):
        return "direct"
    return "DESCONOCIDO"


def leer_csv_seguro(ruta: Path, rename: dict):
    if not ruta.exists():
        print(f"  ⚠  No encontrado: {ruta.name}")
        return None
    return pd.read_csv(ruta, low_memory=False).rename(columns=rename)



In [13]:
# Mapas de renombrado de columnas

RENAME_ADJ_PROV = {
    "Open Contracting ID": "ocid",
    "Entrega compilada:ID de Entrega": "id_entrega",
    "Entrega compilada:Adjudicaciones:ID de Adjudicación": "id_adjudicacion",
    "Entrega compilada:Adjudicaciones:Proveedores:ID de Organización": "id_org_proveedor",
    "Entrega compilada:Adjudicaciones:Proveedores:Nombre de la Organización": "nombre_proveedor",
}

RENAME_ADJUDICACIONES = {
    "Open Contracting ID": "ocid",
    "Entrega compilada:ID de Entrega": "id_entrega",
    "Entrega compilada:Adjudicaciones:ID de Adjudicación": "id_adjudicacion",
    "Entrega compilada:Adjudicaciones:Valor:Monto": "monto_adjudicado",
    "Entrega compilada:Adjudicaciones:Valor:Moneda": "moneda_adjudicacion",
    "Entrega compilada:Adjudicaciones:Fecha de adjudicación": "fecha_adjudicacion",
}

RENAME_CONTRATOS = {
    "Open Contracting ID": "ocid",
    "Entrega compilada:Contratos:ID del Contrato": "id_contrato",
    "Entrega compilada:Contratos:ID de Adjudicación": "id_adjudicacion",
    "Entrega compilada:Contratos:Periodo:Fecha de inicio": "fecha_inicio_contrato",
    "Entrega compilada:Contratos:Periodo:Fecha de fin": "fecha_fin_contrato",
    "Entrega compilada:Contratos:Periodo:Duración (días)": "duracion_contrato_dias",
    "Entrega compilada:Contratos:Valor:Monto": "monto_contrato",
    "Entrega compilada:Contratos:Valor:Moneda": "moneda_contrato",
    "Entrega compilada:Contratos:Fecha de firma": "fecha_firma_contrato",
}

RENAME_LICITANTES = {
    "Open Contracting ID": "ocid",
    "Entrega compilada:Licitación:ID de licitación": "id_licitacion",
    "Entrega compilada:Licitación:Licitantes:ID de Organización": "id_org_licitante",
    "Entrega compilada:Licitación:Licitantes:Nombre de la Organización": "nombre_licitante",
}

RENAME_PARTES = {
    "Open Contracting ID": "ocid",
    "Entrega compilada:Partes involucradas:ID de Entidad": "id_entidad",
    "Entrega compilada:Partes involucradas:Nombre común": "nombre_comun",
    "Entrega compilada:Partes involucradas:Identificador principal:ID": "ruc_raw",
    "Entrega compilada:Partes involucradas:Dirección:Departamento": "departamento",
    "Entrega compilada:Partes involucradas:Roles de las partes": "roles",
}

RENAME_REGISTROS = {
    "Open Contracting ID": "ocid",
    "Entrega compilada:Comprador:ID de Organización": "id_organizacion_comprador",
    "Entrega compilada:Comprador:Nombre de la Organización": "nombre_comprador",
    "Entrega compilada:Licitación:ID de licitación": "id_licitacion",
    "Entrega compilada:Licitación:Detalles del método de contratación": "detalle_metodo",
    "Entrega compilada:Licitación:Categoría principal de contratación": "categoria_principal",
    "compiledRelease/tender/value/amount_PEN": "monto_referencial_pen",
    "Entrega compilada:Licitación:Periodo de licitación:Fecha de inicio": "fecha_inicio_licitacion",
    "Entrega compilada:Licitación:Periodo de licitación:Fecha de fin": "fecha_fin_licitacion",
    "Entrega compilada:Licitación:Número de licitantes": "num_licitantes_reportado",
    "Entrega compilada:Licitación:Método de contratación": "metodo_contratacion",
}

print(" Mapas de renombrado definidos.")



 Mapas de renombrado definidos.


In [14]:
# Lectura y concatenación mensual

acumuladores = {
    "adj_prov":       ([], "Ent_Adj_Proveedores.csv",   RENAME_ADJ_PROV),
    "adjudicaciones": ([], "Ent_Adjudicaciones.csv",     RENAME_ADJUDICACIONES),
    "contratos":      ([], "Ent_Contratos.csv",           RENAME_CONTRATOS),
    "licitantes":     ([], "Ent_Lic_Licitantes.csv",      RENAME_LICITANTES),
    "partes":         ([], "Ent_PartesInvolucradas.csv",  RENAME_PARTES),
    "registros":      ([], "Registros.csv",               RENAME_REGISTROS),
}

for mes in carpetas_meses:
    ruta_mes = RUTA_BASE / mes
    for nombre, (lista, archivo, rename) in acumuladores.items():
        df = leer_csv_seguro(ruta_mes / archivo, rename)
        if df is not None:
            lista.append(df)

tablas = {}
for nombre, (lista, archivo, _) in acumuladores.items():
    if lista:
        tablas[nombre] = pd.concat(lista, ignore_index=True)
        print(f"✅ {archivo:<35} → {tablas[nombre].shape[0]:>8,} filas")
    else:
        print(f"❌ {archivo:<35} → sin datos")

# Verificar tablas críticas
criticas = ["adjudicaciones", "adj_prov", "registros"]
faltantes = [t for t in criticas if t not in tablas]
if faltantes:
    raise RuntimeError(f"Faltan tablas críticas: {faltantes}. Revisa ./usar/")

df_adj_prov       = tablas["adj_prov"]
df_adjudicaciones = tablas["adjudicaciones"]
df_contratos      = tablas.get("contratos")
df_licitantes     = tablas.get("licitantes")
df_partes         = tablas.get("partes")
df_registros      = tablas["registros"]



✅ Ent_Adj_Proveedores.csv             →   65,597 filas
✅ Ent_Adjudicaciones.csv              →   65,708 filas
✅ Ent_Contratos.csv                   →   59,162 filas
✅ Ent_Lic_Licitantes.csv              →  991,101 filas
✅ Ent_PartesInvolucradas.csv          → 1,070,212 filas
✅ Registros.csv                       →   79,109 filas


In [15]:
# Limpieza individual de cada tabla

# ── 4.1  Ent_Adj_Proveedores ──────────────────────────────────────────────────
df_adj_prov = df_adj_prov.copy()
df_adj_prov["ruc_proveedor"] = df_adj_prov["id_org_proveedor"].apply(extraer_ruc)
df_adj_prov = (
    df_adj_prov[["ocid", "id_adjudicacion", "ruc_proveedor", "nombre_proveedor"]]
    .drop_duplicates(subset=["id_adjudicacion", "ruc_proveedor"])
)
print(f"adj_prov limpio:        {df_adj_prov.shape[0]:>8,} filas")

# ── 4.2  Ent_Adjudicaciones ───────────────────────────────────────────────────
df_adjudicaciones = df_adjudicaciones.copy()
df_adjudicaciones = limpiar_fechas(df_adjudicaciones, ["fecha_adjudicacion"])
df_adjudicaciones["monto_adjudicado_pen"] = np.where(
    df_adjudicaciones["moneda_adjudicacion"] == "PEN",
    df_adjudicaciones["monto_adjudicado"],
    np.nan,
)
df_adjudicaciones = df_adjudicaciones[df_adjudicaciones["monto_adjudicado"] > 0].copy()
df_adjudicaciones.drop(columns=["id_entrega", "moneda_adjudicacion"],
                       inplace=True, errors="ignore")
df_adjudicaciones = df_adjudicaciones.drop_duplicates(subset=["id_adjudicacion"])
print(f"adjudicaciones limpio:  {df_adjudicaciones.shape[0]:>8,} filas")

# ── 4.3  Ent_Contratos ────────────────────────────────────────────────────────
if df_contratos is not None:
    df_contratos = df_contratos[[
        "ocid", "id_adjudicacion", "id_contrato",
        "fecha_inicio_contrato", "fecha_fin_contrato",
        "duracion_contrato_dias", "monto_contrato",
        "moneda_contrato", "fecha_firma_contrato"
    ]].copy()
    df_contratos = limpiar_fechas(df_contratos, [
        "fecha_inicio_contrato", "fecha_fin_contrato", "fecha_firma_contrato"
    ])
    # Anular fechas lógicamente invertidas (conservar la fila, solo limpiar fechas)
    mask_inv = df_contratos["fecha_fin_contrato"] < df_contratos["fecha_inicio_contrato"]
    df_contratos.loc[mask_inv, ["fecha_inicio_contrato", "fecha_fin_contrato"]] = pd.NaT
    # Si hay múltiples contratos por adjudicación: conservar el de mayor monto
    df_contratos = (
        df_contratos
        .sort_values("monto_contrato", ascending=False)
        .drop_duplicates(subset=["id_adjudicacion"])
    )
    print(f"contratos limpio:       {df_contratos.shape[0]:>8,} filas")

# ── 4.4  Ent_Lic_Licitantes ───────────────────────────────────────────────────
if df_licitantes is not None:
    df_licitantes = df_licitantes.copy()
    df_licitantes["ruc_licitante"] = df_licitantes["id_org_licitante"].apply(extraer_ruc)
    # Conservamos nombre_licitante para identificar consorcios sin RUC
    df_licitantes = df_licitantes[["ocid", "ruc_licitante", "nombre_licitante"]].copy()
    df_licitantes = df_licitantes.drop_duplicates(subset=["ocid", "ruc_licitante"])
    print(f"licitantes limpio:      {df_licitantes.shape[0]:>8,} filas")

# ── 4.5  Ent_PartesInvolucradas ───────────────────────────────────────────────
if df_partes is not None:
    df_partes = df_partes.copy()
    df_partes["es_comprador"] = df_partes["roles"].str.contains("buyer",    na=False)
    df_partes["es_proveedor"] = df_partes["roles"].str.contains("supplier", na=False)
    df_partes["ruc_parte"]    = df_partes["ruc_raw"].apply(extraer_ruc)
    df_partes = df_partes[[
        "ocid", "ruc_parte", "departamento", "es_comprador", "es_proveedor"
    ]].copy()
    print(f"partes limpio:          {df_partes.shape[0]:>8,} filas")

# ── 4.6  Registros ────────────────────────────────────────────────────────────
df_registros = df_registros.copy()
df_registros["metodo_contratacion_limpio"] = df_registros.apply(inferir_metodo, axis=1)
df_registros["metodo_desconocido"] = (
    df_registros["metodo_contratacion_limpio"] == "DESCONOCIDO"
).astype("Int8")
df_registros = limpiar_fechas(df_registros,
                               ["fecha_inicio_licitacion", "fecha_fin_licitacion"])
df_registros.drop(
    columns=["id_entrega", "detalle_metodo", "metodo_contratacion",
             "id_licitacion", "nombre_comprador"],
    inplace=True, errors="ignore"
)
df_registros = df_registros.drop_duplicates(subset=["ocid"])
print(f"registros limpio:       {df_registros.shape[0]:>8,} filas")
print("\n Limpieza individual completada.")


adj_prov limpio:          65,551 filas
adjudicaciones limpio:    65,660 filas
contratos limpio:         57,749 filas
licitantes limpio:       968,339 filas
partes limpio:          1,070,212 filas
registros limpio:         79,109 filas

 Limpieza individual completada.


In [16]:
# Conteo real de licitantes (reconciliado)

if df_licitantes is not None:
    conteo_lic = (
        df_licitantes
        .groupby("ocid")
        .agg(
            num_licitantes_real=(
                "ruc_licitante",
                lambda x: len(x)   # incluye NaN (consorcios sin RUC)
            ),
            tiene_consorcio_licitante=(
                "ruc_licitante",
                lambda x: (x == "CONSORCIO_SIN_RUC").any()
            ),
        )
        .reset_index()
    )
    df_registros = df_registros.merge(conteo_lic, on="ocid", how="left")
else:
    df_registros["num_licitantes_real"]       = np.nan
    df_registros["tiene_consorcio_licitante"] = False

# Valor final: real > reportado > 1
df_registros["num_licitantes_final"] = (
    df_registros["num_licitantes_real"]
    .fillna(df_registros["num_licitantes_reportado"])
    .fillna(1)
    .astype(int)
)
df_registros["licitantes_dato_faltante"] = (
    df_registros["num_licitantes_real"].isna().astype("Int8")
)
print(" Conteo de licitantes reconciliado.")
print(df_registros[["num_licitantes_real",
                     "num_licitantes_reportado",
                     "num_licitantes_final"]].describe())



 Conteo de licitantes reconciliado.
       num_licitantes_real  num_licitantes_reportado  num_licitantes_final
count         73966.000000              73966.000000          79109.000000
mean             13.091677                 13.222143             12.305578
std              14.273665                 14.704660             14.120179
min               1.000000                  1.000000              1.000000
25%               3.000000                  3.000000              3.000000
50%               9.000000                  9.000000              8.000000
75%              17.000000                 17.000000             16.000000
max             267.000000                282.000000            267.000000


In [17]:
# Extracción de departamentos (sin duplicados)

if df_partes is not None:
    # Un departamento por OCID (comprador)
    dept_comprador = (
        df_partes[df_partes["es_comprador"]]
        .dropna(subset=["departamento"])
        .groupby("ocid")["departamento"]
        .first()
        .reset_index()
        .rename(columns={"departamento": "dept_entidad"})
    )
    # Un departamento por (ocid, ruc_proveedor): tomamos la moda para evitar duplicados
    dept_proveedor = (
        df_partes[df_partes["es_proveedor"] & df_partes["ruc_parte"].notna()]
        .groupby(["ocid", "ruc_parte"])["departamento"]
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
        .reset_index()
        .rename(columns={"ruc_parte": "ruc_proveedor", "departamento": "dept_proveedor"})
    )
else:
    dept_comprador = pd.DataFrame(columns=["ocid", "dept_entidad"])
    dept_proveedor = pd.DataFrame(columns=["ocid", "ruc_proveedor", "dept_proveedor"])

print(f" dept_comprador: {dept_comprador.shape[0]:,} OCIDs con departamento")
print(f" dept_proveedor: {dept_proveedor.shape[0]:,} pares (ocid, ruc_proveedor)")




 dept_comprador: 79,109 OCIDs con departamento
 dept_proveedor: 58,815 pares (ocid, ruc_proveedor)


In [18]:
# CELDA 7 — Construcción de la tabla base (joins con assertions)

# Orden: adjudicaciones → proveedor ganador → registros → contratos → dptos.
# Assertions después de cada join: si algo multiplica filas, el error
# aparece inmediatamente con la cifra exacta.
# ─────────────────────────────────────────────────────────────────────────────

n0 = df_adjudicaciones.shape[0]

# 1. Adjudicaciones + proveedor ganador
df_base = df_adjudicaciones.merge(
    df_adj_prov[["id_adjudicacion", "ruc_proveedor", "nombre_proveedor"]],
    on="id_adjudicacion",
    how="left",
)
assert df_base.shape[0] == n0, (
    f"⚠ Join adj_prov multiplicó filas: {n0} → {df_base.shape[0]}"
)

# 2. + Registros (contexto de la licitación)
cols_reg = [
    "ocid", "id_organizacion_comprador",
    "monto_referencial_pen", "fecha_inicio_licitacion", "fecha_fin_licitacion",
    "num_licitantes_final", "licitantes_dato_faltante", "tiene_consorcio_licitante",
    "metodo_contratacion_limpio", "metodo_desconocido", "categoria_principal",
]
df_base = df_base.merge(
    df_registros[[c for c in cols_reg if c in df_registros.columns]],
    on="ocid",
    how="left",
)
assert df_base.shape[0] == n0, (
    f"⚠ Join registros multiplicó filas: {n0} → {df_base.shape[0]}"
)

# 3. + Contratos (left join: puede ser NaN si aún no se firmó)
if df_contratos is not None:
    cols_cont = [
        "id_adjudicacion", "id_contrato",
        "fecha_inicio_contrato", "fecha_fin_contrato",
        "duracion_contrato_dias", "monto_contrato", "moneda_contrato",
    ]
    df_base = df_base.merge(
        df_contratos[cols_cont],
        on="id_adjudicacion",
        how="left",
    )
    assert df_base.shape[0] == n0, (
        f"⚠ Join contratos multiplicó filas: {n0} → {df_base.shape[0]}"
    )

# 4. + Departamentos
df_base = df_base.merge(dept_comprador, on="ocid",                     how="left")
df_base = df_base.merge(dept_proveedor, on=["ocid", "ruc_proveedor"],  how="left")
assert df_base.shape[0] == n0, (
    f"⚠ Join departamentos multiplicó filas: {n0} → {df_base.shape[0]}"
)

print(f" Tabla base: {df_base.shape[0]:,} filas × {df_base.shape[1]} columnas")




 Tabla base: 65,660 filas × 26 columnas


In [19]:
# Variables derivadas e indicadores de riesgo
#
# NOTA: Las columnas alerta_* son flags de auditoría derivados de otras
# columnas. Son útiles para reportes pero NO deben usarse como features
# en ML (multicolinealidad). Se excluyen en la matriz_modelo.
# ─────────────────────────────────────────────────────────────────────────────

# Días entre cierre de licitación y adjudicación
df_base["dias_plazo"] = (
    df_base["fecha_adjudicacion"] - df_base["fecha_fin_licitacion"]
).dt.days

# Porcentaje del monto adjudicado sobre el referencial
df_base["porc_adjudicado"] = np.where(
    df_base["monto_referencial_pen"].notna() & (df_base["monto_referencial_pen"] > 0),
    (df_base["monto_adjudicado_pen"] / df_base["monto_referencial_pen"]) * 100,
    np.nan,
)

# Proveedor fuera de la región del comprador
# → float con NaN cuando falta alguno de los dos departamentos
df_base["proveedor_fuera_region"] = np.where(
    df_base["dept_entidad"].isna() | df_base["dept_proveedor"].isna(),
    np.nan,
    (df_base["dept_entidad"] != df_base["dept_proveedor"]).astype(float),
)

# Consorcio ganador
df_base["es_consorcio"] = (
    df_base["nombre_proveedor"].str.upper().str.contains("CONSORCIO", na=False)
    | (df_base["ruc_proveedor"] == "CONSORCIO_SIN_RUC")
).astype("Int8")

# Contrato firmado
if "id_contrato" in df_base.columns:
    df_base["tiene_contrato"] = df_base["id_contrato"].notna().astype("Int8")
else:
    df_base["tiene_contrato"] = pd.NA

# Duración del contrato: rellenar con diferencia de fechas si el campo está vacío
if "duracion_contrato_dias" in df_base.columns:
    duracion_calc = (
        df_base["fecha_fin_contrato"] - df_base["fecha_inicio_contrato"]
    ).dt.days
    df_base["duracion_contrato_dias"] = df_base["duracion_contrato_dias"].fillna(duracion_calc)

# Alertas de auditoría (flags binarios)
df_base["alerta_sobrecosto"]      = (df_base["porc_adjudicado"] > 120).astype("Int8")
df_base["alerta_plazo_corto"]     = (df_base["dias_plazo"].between(-1, 2)).astype("Int8")
df_base["alerta_sin_competencia"] = (df_base["num_licitantes_final"] == 1).astype("Int8")
df_base["alerta_proveedor_lejano"] = df_base["proveedor_fuera_region"].astype("Int8")

print(" Variables derivadas generadas.")
print(df_base[["dias_plazo", "porc_adjudicado",
               "proveedor_fuera_region", "es_consorcio"]].describe())

 Variables derivadas generadas.
         dias_plazo  porc_adjudicado  proveedor_fuera_region  es_consorcio
count  61039.000000     62530.000000                     0.0       65660.0
mean      31.608218        89.278942                     NaN      0.234709
std       37.453170        47.195269                     NaN       0.42382
min     -278.000000         0.000215                     NaN           0.0
25%       13.000000        85.626398                     NaN           0.0
50%       20.000000        98.034889                     NaN           0.0
75%       38.000000       100.000000                     NaN           0.0
max      436.000000     10000.000000                     NaN           1.0


In [20]:
# Agregados históricos por proveedor (sin data leakage)
#
# Usamos expanding window + shift(1) para que cada fila solo "vea"
# los contratos ANTERIORES del mismo proveedor.
# Sin shift(1), el modelo vería la fila actual en sus propios agregados
# → data leakage.
# ─────────────────────────────────────────────────────────────────────────────

df_base = df_base.sort_values("fecha_adjudicacion").reset_index(drop=True)

# Número de contratos previos del proveedor (0 para el primero)
df_base["num_contratos_previos"] = (
    df_base.groupby("ruc_proveedor").cumcount()
)

# Monto acumulado previo
df_base["monto_acum_proveedor"] = (
    df_base.groupby("ruc_proveedor")["monto_adjudicado_pen"]
    .expanding()
    .sum()
    .shift(1)
    .reset_index(level=0, drop=True)
)

# Monto promedio previo
df_base["monto_promedio_previo"] = (
    df_base.groupby("ruc_proveedor")["monto_adjudicado_pen"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

print(" Agregados históricos sin data leakage calculados.")
print(df_base[["num_contratos_previos",
               "monto_acum_proveedor",
               "monto_promedio_previo"]].describe())

 Agregados históricos sin data leakage calculados.
       num_contratos_previos  monto_acum_proveedor  monto_promedio_previo
count           64187.000000          6.389300e+04           6.389300e+04
mean             1793.137240          4.987713e+09           1.054454e+06
std              3852.072901          1.118006e+10           5.353200e+06
min                 0.000000          1.000000e+00           1.000000e+00
25%                 0.000000          1.828620e+05           1.077485e+05
50%                 2.000000          9.028395e+05           2.529827e+05
75%                55.000000          4.346787e+07           1.956745e+06
max             15148.000000          4.817628e+10           1.111909e+09


In [21]:
# Construcción de nodos y aristas para grafo
#
# PROBLEMA del código anterior: guardaba todo en una tabla plana y la
# llamaba "tabla para grafos", lo cual era incorrecto. Un grafo necesita:
#   • Nodos: entidades únicas (compradores y proveedores)
#   • Aristas: relaciones entre ellos (contratos/adjudicaciones)
#
# Aquí generamos las tres tablas correctas para construir el grafo
# con NetworkX, Neo4j o cualquier herramienta de análisis de redes.
# ─────────────────────────────────────────────────────────────────────────────

# ── Nodos compradores ─────────────────────────────────────────────────────────
nodos_compradores = (
    df_base[["id_organizacion_comprador", "dept_entidad"]]
    .dropna(subset=["id_organizacion_comprador"])
    .drop_duplicates(subset=["id_organizacion_comprador"])
    .rename(columns={"id_organizacion_comprador": "id_nodo"})
    .assign(tipo_nodo="comprador")
)

# ── Nodos proveedores ─────────────────────────────────────────────────────────
nodos_proveedores = (
    df_base[["ruc_proveedor", "nombre_proveedor", "dept_proveedor", "es_consorcio"]]
    .dropna(subset=["ruc_proveedor"])
    .drop_duplicates(subset=["ruc_proveedor"])
    .rename(columns={"ruc_proveedor": "id_nodo", "dept_proveedor": "dept_entidad"})
    .assign(tipo_nodo="proveedor")
)

# Unir en una sola tabla de nodos (id_nodo es la llave en el grafo)
nodos_grafo = pd.concat(
    [nodos_compradores, nodos_proveedores],
    ignore_index=True
)

# ── Aristas (contratos/adjudicaciones) ────────────────────────────────────────
cols_aristas = [
    "id_adjudicacion",
    "id_organizacion_comprador",   # origen del arco
    "ruc_proveedor",               # destino del arco
    "monto_adjudicado_pen",
    "fecha_adjudicacion",
    "metodo_contratacion_limpio",
    "num_licitantes_final",
    "es_consorcio",
    "proveedor_fuera_region",
    "dias_plazo",
    "porc_adjudicado",
    "alerta_sobrecosto",
    "alerta_plazo_corto",
    "alerta_sin_competencia",
    "alerta_proveedor_lejano",
]
aristas_grafo = (
    df_base[[c for c in cols_aristas if c in df_base.columns]]
    .dropna(subset=["id_organizacion_comprador", "ruc_proveedor"])
    .rename(columns={
        "id_organizacion_comprador": "origen",
        "ruc_proveedor": "destino",
    })
)

print(f" Grafo construido:")
print(f"   Nodos compradores : {nodos_compradores.shape[0]:>8,}")
print(f"   Nodos proveedores : {nodos_proveedores.shape[0]:>8,}")
print(f"   Total nodos       : {nodos_grafo.shape[0]:>8,}")
print(f"   Aristas (contratos): {aristas_grafo.shape[0]:>8,}")

 Grafo construido:
   Nodos compradores :    2,804
   Nodos proveedores :   20,566
   Total nodos       :   23,370
   Aristas (contratos):   64,187


In [22]:
# Guardado de archivos finales
#
# Salidas:
#   adjudicaciones_procesadas.parquet  → tabla completa (auditoría + reglas)
#   nodos_grafo.parquet                → entidades únicas para el grafo
#   aristas_grafo.parquet              → contratos como aristas del grafo
#   matriz_modelo.parquet              → solo features numéricas/categóricas
#                                        para ML, sin IDs ni alertas derivadas
# ─────────────────────────────────────────────────────────────────────────────

EXCLUIR_MODELO = {
    # Identificadores
    "ocid", "id_adjudicacion", "id_contrato",
    "id_organizacion_comprador", "ruc_proveedor",
    # Texto libre
    "nombre_proveedor",
    # Alertas derivadas (multicolinealidad en ML)
    "alerta_sobrecosto", "alerta_plazo_corto",
    "alerta_sin_competencia", "alerta_proveedor_lejano",
    # Geografía textual (ya codificada en proveedor_fuera_region)
    "dept_entidad", "dept_proveedor",
    # Moneda (constante en la mayoría de registros)
    "moneda_contrato",
    # Fechas crudas (ya usadas para calcular días)
    "fecha_adjudicacion", "fecha_inicio_licitacion", "fecha_fin_licitacion",
    "fecha_inicio_contrato", "fecha_fin_contrato", "fecha_firma_contrato",
}

cols_modelo = [c for c in df_base.columns if c not in EXCLUIR_MODELO]
df_modelo   = df_base[cols_modelo].copy()

# Guardar todo
df_base.to_parquet("adjudicaciones_procesadas.parquet", index=False)
nodos_grafo.to_parquet("nodos_grafo.parquet",           index=False)
aristas_grafo.to_parquet("aristas_grafo.parquet",       index=False)
df_modelo.to_parquet("matriz_modelo.parquet",           index=False)

print(" Archivos guardados:")
print(f"   adjudicaciones_procesadas : {df_base.shape[0]:>8,} filas × {df_base.shape[1]:>3} cols")
print(f"   nodos_grafo               : {nodos_grafo.shape[0]:>8,} filas × {nodos_grafo.shape[1]:>3} cols")
print(f"   aristas_grafo             : {aristas_grafo.shape[0]:>8,} filas × {aristas_grafo.shape[1]:>3} cols")
print(f"   matriz_modelo             : {df_modelo.shape[0]:>8,} filas × {df_modelo.shape[1]:>3} cols")
print("\nColumnas en matriz_modelo:")
print(df_modelo.dtypes.to_string())

 Archivos guardados:
   adjudicaciones_procesadas :   65,660 filas ×  38 cols
   nodos_grafo               :   23,370 filas ×   5 cols
   aristas_grafo             :   64,187 filas ×  15 cols
   matriz_modelo             :   65,660 filas ×  20 cols

Columnas en matriz_modelo:
monto_adjudicado                                           float64
Entrega compilada:Adjudicaciones:Valor:Nombre de Moneda     object
monto_adjudicado_pen                                       float64
monto_referencial_pen                                      float64
num_licitantes_final                                         int64
licitantes_dato_faltante                                      Int8
tiene_consorcio_licitante                                   object
metodo_contratacion_limpio                                  object
metodo_desconocido                                            Int8
categoria_principal                                         object
duracion_contrato_dias                               

In [23]:
df = pd.read_parquet("matriz_modelo.parquet")
print(df.shape)
print(df.isnull().sum())

(65660, 20)
monto_adjudicado                                               0
Entrega compilada:Adjudicaciones:Valor:Nombre de Moneda        0
monto_adjudicado_pen                                        1490
monto_referencial_pen                                          0
num_licitantes_final                                           0
licitantes_dato_faltante                                       0
tiene_consorcio_licitante                                     10
metodo_contratacion_limpio                                     0
metodo_desconocido                                             0
categoria_principal                                            0
duracion_contrato_dias                                      8079
monto_contrato                                              8079
dias_plazo                                                  4621
porc_adjudicado                                             3130
proveedor_fuera_region                                     65660
es_consorcio 